In [9]:
import pandas as pd
df = pd.read_csv('Global_Superstore2.csv', encoding='latin1')
print(df.shape)
df.head()
df.columns



(51290, 24)


Index(['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode',
       'Customer ID', 'Customer Name', 'Segment', 'City', 'State', 'Country',
       'Postal Code', 'Market', 'Region', 'Product ID', 'Category',
       'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount',
       'Profit', 'Shipping Cost', 'Order Priority'],
      dtype='object')

Star Schema Design:

Fact Table:
fact_sales(order_id, customer_id, product_id, region_id, order_date_id, ship_date_id, sales, quantity, discount, profit, shipping_cost)

Dimension Tables:
dim_customer(customer_id, customer_name, segment)
dim_product(product_id, product_name, category, sub_category)
dim_region(region_id, city, state, country, market, region)
dim_date(date_id, full_date, year, month, day)



In [10]:
dim_customer = df[['Customer ID', 'Customer Name', 'Segment']].drop_duplicates().reset_index(drop=True)

dim_customer.head()


,Customer ID,Customer Name,Segment
0,RH-19495,Rick Hansen,Consumer
1,JR-16210,Justin Ritter,Corporate
2,CR-12730,Craig Reiter,Consumer
3,KM-16375,Katherine Murray,Home Office
4,RH-9495,Rick Hansen,Consumer


In [11]:
dim_region = df[['City', 'State', 'Country', 'Market', 'Region']].drop_duplicates().reset_index(drop=True)

dim_region.head()


,City,State,Country,Market,Region
0,New York City,New York,United States,US,East
1,Wollongong,New South Wales,Australia,APAC,Oceania
2,Brisbane,Queensland,Australia,APAC,Oceania
3,Berlin,Berlin,Germany,EU,Central
4,Dakar,Dakar,Senegal,Africa,Africa


In [12]:
df['Order Date'] = pd.to_datetime(df['Order Date'])
df['Ship Date'] = pd.to_datetime(df['Ship Date'])

dim_date = pd.DataFrame({
    'full_date': pd.concat([df['Order Date'], df['Ship Date']]).drop_duplicates()
})

dim_date['year'] = dim_date['full_date'].dt.year
dim_date['month'] = dim_date['full_date'].dt.month
dim_date['day'] = dim_date['full_date'].dt.day

dim_date = dim_date.reset_index(drop=True)

dim_date.head()


/tmp/ipython-input-438/1632025268.py:1: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Order Date'] = pd.to_datetime(df['Order Date'])
/tmp/ipython-input-438/1632025268.py:2: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Ship Date'] = pd.to_datetime(df['Ship Date'])


,full_date,year,month,day
0,2012-07-31,2012,7,31
1,2013-02-05,2013,2,5
2,2013-10-17,2013,10,17
3,2013-01-28,2013,1,28
4,2013-11-05,2013,11,5


In [13]:
dim_product = df[['Product ID', 'Product Name', 'Category', 'Sub-Category']] \
                .drop_duplicates() \
                .reset_index(drop=True)

In [14]:
dim_customer['customer_key'] = dim_customer.index + 1
dim_product['product_key'] = dim_product.index + 1
dim_region['region_key'] = dim_region.index + 1
dim_date['date_key'] = dim_date.index + 1


In [15]:

df_fact = df.merge(dim_customer, on=['Customer ID', 'Customer Name', 'Segment'], how='left')


df_fact = df_fact.merge(dim_product, on=['Product ID', 'Product Name', 'Category', 'Sub-Category'], how='left')


df_fact = df_fact.merge(dim_region, on=['City', 'State', 'Country', 'Market', 'Region'], how='left')


df_fact = df_fact.merge(dim_date[['full_date', 'date_key']],
                        left_on='Order Date', right_on='full_date', how='left')

df_fact.rename(columns={'date_key': 'order_date_key'}, inplace=True)

df_fact = df_fact.merge(dim_date[['full_date', 'date_key']],
                        left_on='Ship Date', right_on='full_date', how='left')

df_fact.rename(columns={'date_key': 'ship_date_key'}, inplace=True)


In [16]:
fact_sales = df_fact[[
    'Order ID',
    'customer_key',
    'product_key',
    'region_key',
    'order_date_key',
    'ship_date_key',
    'Sales',
    'Quantity',
    'Discount',
    'Profit',
    'Shipping Cost'
]]

fact_sales.head()


,Order ID,customer_key,product_key,region_key,order_date_key,ship_date_key,Sales,Quantity,Discount,Profit,Shipping Cost
0,CA-2012-124891,1,1,1,1,1,2309.650,7,0.0,762.1845,933.57
1,IN-2013-77878,2,2,2,2,943,3709.395,9,0.1,-288.7650,923.63
2,IN-2013-71249,3,3,3,3,429,5175.171,9,0.1,919.9710,915.49
3,ES-2013-1579342,4,4,4,4,999,2892.510,5,0.1,-96.5400,910.16
4,SG-2013-4320,5,5,5,5,936,2832.960,8,0.0,311.5200,903.04


In [17]:
fact_sales.shape
fact_sales.isnull().sum()


,0
Order ID,0
customer_key,0
product_key,0
region_key,0
order_date_key,0
ship_date_key,0
Sales,0
Quantity,0
Discount,0
Profit,0


In [18]:
dim_customer.to_csv("dim_customer.csv", index=False)
dim_product.to_csv("dim_product.csv", index=False)
dim_region.to_csv("dim_region.csv", index=False)
dim_date.to_csv("dim_date.csv", index=False)
fact_sales.to_csv("fact_sales.csv", index=False)


In [19]:
analysis1 = fact_sales.groupby('product_key')['Sales'].sum().reset_index()
analysis2 = fact_sales.groupby('region_key')['Profit'].sum().reset_index()

analysis_outputs = pd.concat([analysis1, analysis2], axis=1)
analysis_outputs.to_csv("analysis_outputs.csv", index=False)
